# Ambiguous folds: Arcimboldo against the baseline

A new goal for the crane. We're no longer trying to hide the wings' pictures in the flat sheet. Instead, the flat sheet should read as one scene, where every piece looks like part of that scene. Folding then cuts a piece out of its context, and it resolves into something else. Dalí's double images work this way: context decides what a shape is.

The prompts: the Tokyo skyline under sakura and an orange sky when flat, a red sun on one wing, and a cicada on the other.

**What folding does to a wing.** Each crane wing is an upright triangle made of two shards of the sheet. On the flat sheet, those shards lie along two different edges, and they touch only at a corner tip. Folding does three things to them at once:

- **cuts** them out of their surroundings
- **turns** each by a different angle
- **joins** them along an edge that was never shared on the sheet

**The two methods.** Each is a set of views of the sheet, each view with its own prompt. They're sampled together as in [MultiDiffusion](https://arxiv.org/abs/2302.08113) (Bar-Tal et al., ICML 2023): at every denoising step, every view's prediction is mapped back onto the sheet and averaged per pixel.

| Method | Views and prompts | The hypothesis |
| --- | --- | --- |
| `average` (baseline) | flat sheet: Tokyo. Folded views: red sun, cicada. Plain average | what we had |
| `arcimboldo` | the Tokyo prompt controls only the low frequencies. The folded views, and nine overlapping windows that each ask for a close-up of the scene, control the high frequencies | the scene is a large gestalt made of small objects, and folding pulls out one object |

**Why only these two.** The first run compared four methods (`context`, `arcimboldo`, `silhouette` and `fold-in`). Arcimboldo gave the best result by far: the cicada is drawn with the same branch strokes and blossoms as the rest of the scene, so it belongs to the flat sheet instead of sitting on top of it. The other methods made the cicada either too faint or tacked on. They're still in `ambiguous.py`, and `ambiguous.METHODS` lists them, if you want to try one again.

## Setup

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("ambiguous.py"):
    if not os.path.exists("folded-illusions"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/folded-illusions.git"], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "decal", "illusion", "factorized", "ambiguous"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from fold_map import FoldedModel
from decal import test_picture
from illusion import FoldViews, canvas_picture, plot_result, printable, square_picture
import factorized
import ambiguous

## The crane's wings, in place and folded

Each row shows the flat sheet, then for each wing its two shards in place on the flat sheet with their surroundings, then the folded view. The letter is a stand-in picture. In place, the shards are just part of the A's background. Folded, they become the crane's body.

In [ ]:
model = FoldedModel("004_traditional_Crane_cp.fold", "004_traditional_Crane_state.fold")
views64, views256 = FoldViews(model, 64), FoldViews(model, 256)

test_sheet = square_picture(test_picture("A", (60, 140, 90), (1000, 1000)), 256)
ambiguous.plot_reveal(views256, {"test picture": test_sheet})
plt.show()
for side in ("front", "back"):
    print(side, "wing shards:", [int(m.sum()) for m in ambiguous.wing_shards(views256, side)], "sheet pixels each")

## What each method shows the model

Every tile below is one view: what the model is shown for it at every denoising step, with its prompt key and weight. Gray is the plain backdrop. Faded pixels have less say in the blend. Views marked low or high control only those frequencies. Read this to see what each method asks for before spending GPU time.

`CONTEXT_WEIGHT` is the windows' total say at every pixel. Each window fades out across its overlap with its neighbors, so together they add up to the same weight everywhere and leave no seams. 1.0 gives them as much say as the folded views.

In [ ]:
METHODS = ("average", "arcimboldo")
CONTEXT_WEIGHT = 1.0

def views_for(method, fv):
    return ambiguous.method_views(method, fv, context_weight=CONTEXT_WEIGHT)

test64 = square_picture(test_picture("A", (60, 140, 90), (1000, 1000)), 64)
for method in METHODS:
    fig = ambiguous.plot_views(views_for(method, views64), test64)
    fig.suptitle(f"{method}: {len(views_for(method, views64))} views", y=1.02)
    plt.show()

## Generate (needs a GPU)

Setup matches the other notebooks.

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the licenses for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
# protobuf 7.36 or later: current transformers ships a tokenizer file older protobuf can't read.
!pip install -q -U diffusers transformers accelerate sentencepiece "protobuf>=7.36"
import sys
if "google.protobuf" in sys.modules:
    print("protobuf was already loaded before this upgrade: choose Runtime → Restart session, then run from the top.")
from huggingface_hub import notebook_login
notebook_login()

### Prompts

- **`FLAT_PROMPT`, `FRONT_PROMPT`, `BACK_PROMPT`:** the flat sheet, the front wing and the back wing.
- **`CONTEXT_PROMPT`:** what a small piece of the flat scene should look like up close, for arcimboldo's windows. It's built from the flat scene's own elements, so a piece that obeys it belongs in the scene.

**Write the wing prompts in terms of edges and texture.** Arcimboldo gives the wings only the high frequencies: anything smaller than about an eighth of the sheet. A wing is about 16 × 32 pixels at stage 1, so a cicada's body and wings are fine detail and come through. A plain sunset doesn't: it's a smooth gradient, which is all low frequency, and the flat sheet is already an orange sky. So the sun is a crisp red disc, whose edge is fine detail, against pale sky with thin cloud bands, which are fine detail too.

`CLIP_TEXTS` are what the scoring checks each view for.

In [ ]:
STYLE = "an oil painting of "
FLAT_PROMPT = STYLE + "the Tokyo skyline at dusk, framed by blooming sakura trees, under a glowing orange and white sky"
FRONT_PROMPT = STYLE + "a large crimson red sun, a perfect crisp-edged disc like the one on the Japanese flag, low in the picture against a pale white sky streaked with thin horizontal bands of cloud"
BACK_PROMPT = STYLE + "a black silhouette of a single cicada seen from directly above, head at the bottom, its folded wings tapering to a point at the top, against a plain glowing orange sky"
CONTEXT_PROMPT = STYLE + "a close-up of dark sakura branches and Tokyo rooftops silhouetted against a glowing orange and white sky, with paper lanterns"

PROMPTS = dict(flat=FLAT_PROMPT, front=FRONT_PROMPT, back=BACK_PROMPT, context=CONTEXT_PROMPT)
VIEW_TITLES = ("flat: Tokyo", "red sun wing", "cicada wing")
CLIP_TEXTS = ("the Tokyo skyline with sakura trees", "a red sun", "a cicada")

### Encode the prompts

Rerun this cell whenever you change the prompts.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

# The safety checker and watermarker aren't used by our sampler, and the safety checker
# doesn't load with current versions of transformers, so skip them.
skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
embeds = ambiguous.encode_prompts(encoder, PROMPTS)

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("encoded:", ", ".join(embeds))

### Load the image models

This only needs to run once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

### Run both methods

Both use the same seed, so differences come from the method alone. Each row shows the flat sheet, then each wing's shards in place on the flat sheet, then the folded view. The question for every row: do the shards look like part of the scene in place, and like the red sun or the cicada once folded?

On a T4, the baseline takes a minute or two, and `arcimboldo` about four times as long, because of its nine windows. If the GPU runs out of memory, lower `BATCH_SIZE`.

`SIGMA_FRAC` sets where `arcimboldo` splits frequencies, as a fraction of the sheet's side. It's much larger than in `factorized_illusions.ipynb`, because the flat prompt only needs to set the large shapes here. If the sun still washes out, try 1/12: the wings then control somewhat larger shapes, so the sun and cicada get bolder but may look more pasted on.

In [ ]:
import time

SIGMA_FRAC = {"arcimboldo": 1 / 8}
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
BATCH_SIZE = 4
SEED = 0

settings = dict(guidance=GUIDANCE, steps=STEPS, batch_size=BATCH_SIZE)
results = {}

def run(method, seed):
    start = time.time()
    out = ambiguous.generate(stage_1, stage_2, views_for(method, views64), views_for(method, views256), embeds,
                             sigma_frac=SIGMA_FRAC.get(method, 0), seed=seed, **settings)
    print(f"{method}, seed {seed}: {time.time() - start:.0f} s")
    return out

for method in METHODS:
    results[f"seed {SEED}, {method}"] = run(method, SEED)

ambiguous.plot_reveal(views256, {k: s2 for k, (_, s2) in results.items()}, VIEW_TITLES)
plt.show()

**What to look for:**

- **Flat:** does it read as one Tokyo scene, without crane-shaped patches?
- **Shards in place:** do they look like sky, branches or rooftops, with nothing sun- or cicada-like?
- **Folded:** does each wing read as its subject? The whole wing is about 16 × 32 pixels at stage 1, so look for a clear shape and color, not detail.

Stage 1 decides each picture's layout, so its results are worth a look too:

In [ ]:
ambiguous.plot_reveal(views64, {k: s1 for k, (s1, _) in results.items()}, VIEW_TITLES)
plt.show()

### Try more seeds

Seeds matter as much as methods, so run `arcimboldo` on a few more. These are added to `results`, so the scoring below ranks them too.

In [ ]:
SEEDS = (1, 2)
MORE = ("arcimboldo",)               # the methods to repeat

for seed in SEEDS:
    batch = {f"seed {seed}, {method}": run(method, seed) for method in MORE}
    results.update(batch)
    ambiguous.plot_reveal(views256, {k: s2 for k, (_, s2) in batch.items()}, VIEW_TITLES)
    plt.show()

### Score every result with CLIP

[CLIP](https://huggingface.co/openai/clip-vit-large-patch14) measures how well an image matches a piece of text. Hiding no longer counts. What counts is whether folding changes what the wing's pixels are. For each result:

- **flat, front, back:** how well the flat sheet and each folded view match their own subject. Higher is better.
- **front_switch, back_switch:** the reveal. This is the folded view's match to its subject, minus how much its shards, in place on the flat sheet, already match that subject. Higher means folding turns the pixels into the subject. Near zero means the subject was already visible flat, or never appeared.
- **score:** the mean of flat, front and back, plus the mean switch.

Differences of a few hundredths are noise. Use the scores to narrow the field, then pick by eye. The first run downloads CLIP, about 1.7 GB.

In [ ]:
if "embed_images" not in globals():
    embed_images, embed_texts = factorized.clip_embedders()
ranking = ambiguous.score(views256, {k: s2 for k, (_, s2) in results.items()}, embed_images, embed_texts, CLIP_TEXTS)

print(f"{'result':28} {'flat':>6} {'front':>6} {'back':>6} {'f_sw':>6} {'b_sw':>6} {'score':>6}")
for r in ranking:
    print(f"{r['label']:28} {r['flat']:6.3f} {r['front']:6.3f} {r['back']:6.3f} "
          f"{r['front_switch']:6.3f} {r['back_switch']:6.3f} {r['score']:6.3f}")

### Fold the one you like

By default this takes CLIP's top result. Set `BEST` to any label from `results` if your eye disagrees, e.g. `"seed 1, arcimboldo"`. This folds the result virtually, saves a print file and downloads it. Print at **actual size**, and fold with the **printed side out**: corner A becomes the tail, C the head, and B and D the wing tips.

Download before disconnecting the runtime, because `results` lives only in its memory.

In [ ]:
BEST = ranking[0]["label"]

sheet = results[BEST][1]
plot_result(views256, sheet, BEST)
plt.show()

name = "ambiguous_" + BEST.replace("seed ", "seed").replace(", ", "_").replace(" ", "") + ".png"
printable(sheet).save(name, dpi=(300, 300))
print("saved", name)
try:
    from google.colab import files
    files.download(name)
except ImportError:
    pass

## Notes

- **Averaging is MultiDiffusion's fusion.** MultiDiffusion fuses overlapping views by least squares, which for per-pixel weights is a weighted average of each view's prediction. The `average` method is exactly `illusion.generate`'s blend with `fold_weight` 0.5, and a test checks that.
- **Why arcimboldo suits a cicada.** The Tokyo view alone decides where the big dark and light masses go. The wing views and windows decide only the strokes inside them. So the cicada gets drawn with the same black branch strokes and blossom clusters as the rest of the scene. In the plain average, the cicada also gets a say over the large shapes, so it shows up as its own dark blob.
- **Windows are shown unrotated and at the sheet's scale,** on the same gray backdrop as the folded views. Only folding turns pixels, which is what makes these anagrams: the same pixels, upright in place and turned on the wing.
- **`CONTEXT_PROMPT` is a design choice,** not part of the method. It decides what the wing's pixels pretend to be up close on the flat sheet.
- **The switch score crops each shard** with a margin of the scene around it, padded to a square with white. The margin is deliberate: it's what a viewer sees on the flat sheet.
- **Windows fade out at their edges.** Every view is drawn on a gray backdrop, so the model draws a window's edge. In the first run, unfaded windows left straight seams across the flat sheet. Fading each window across its overlap with its neighbors, as Mixture of Diffusers does with its tiles, makes the edge count for nothing.
- **`CONTEXT_WEIGHT` and `SIGMA_FRAC` are untuned.**
- **The other methods** (`context`, `silhouette`, `fold-in`) are in `ambiguous.py`. Add one to `METHODS` to run it, and add any prompt it needs: `silhouette` needs `front_shape` and `back_shape`.